In [ ]:
# Numpy 手撕 MLP 核心逻辑
import numpy as np

# 假设 X: (batch, input_dim), W1: (input_dim, hidden_dim), W2: (hidden_dim, output_dim)
def mlp_forward_backward(X, Y, W1, b1, W2, b2, lr=0.1):
    m = X.shape[0] # batch size
    
    # === 前向传播 ===
    Z1 = np.dot(X, W1) + b1
    A1 = np.maximum(0, Z1) # ReLU 激活
    Z2 = np.dot(A1, W2) + b2
    # 假设用 Softmax + CrossEntropy，这里的 A2 为预测概率
    A2 = softmax(Z2) 
    
    # === 反向传播 (核心肌肉记忆) ===
    dZ2 = A2 - Y                           # 输出层误差
    dW2 = np.dot(A1.T, dZ2) / m            # W2 的梯度 (注意 A1 转置)
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m
    
    dA1 = np.dot(dZ2, W2.T)                # 误差反传给隐藏层
    dZ1 = dA1 * (Z1 > 0)                   # 乘上 ReLU 的导数
    dW1 = np.dot(X.T, dZ1) / m             # W1 的梯度 (注意 X 转置)
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m
    
    # === 参数更新 ===
    W1 -= lr * dW1
    W2 -= lr * dW2
    return W1, W2

In [ ]:
# 纯手工 2D 卷积
def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j: j + w] * K).sum()
    return Y

In [ ]:
# 经典 MLP
import torch 
from torch import nn

mlp_net = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

In [ ]:
# 现代 CNN 框架
cnn_net = nn.Sequential(
    # 提特征：无脑 3x3 卷积，通道翻倍，池化砍半尺寸
    nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2), 
    
    nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    
    # 展平与漏斗式分类器
    nn.Flatten(),
    nn.Linear(128 * 7 * 7, 1024), nn.ReLU(),
    nn.Linear(1024, 256), nn.ReLU(),
    nn.Linear(256, 10)
)

In [ ]:
# ResNet的forward
class Residual(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv3 = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else None

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))   # conv→BN→ReLU
        Y = self.bn2(self.conv2(Y))           # conv→BN（先不加 ReLU）
        if self.conv3:
            X = self.conv3(X)                 # 捷径：通道数不同时对齐
        return F.relu(Y + X)                  # 相加 → ReLU